# Exercise 4.3.14 — compute suppressed attention

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `4.3 Interpreting Reasoning Models`  
**Notebook:** `4.3_Interpreting_Reasoning_Models_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=4.3.14](https://delta-drills.vercel.app/?arena_exercise=4.3.14)


# [4.3] Interpreting Reasoning: Thought Anchors (exercises)

> **ARENA [Streamlit Page](https://arena-chapter4-alignment-science.streamlit.app/03_[4.3]_Interpreting_Reasoning_Models)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter4_alignment_science/exercises/part3_interpreting_reasoning_models/4.3_Interpreting_Reasoning_Models_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter4_alignment_science/exercises/part3_interpreting_reasoning_models/4.3_Interpreting_Reasoning_Models_solutions.ipynb?t=20260329)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/header-64.png" width="350">

# Introduction

So far in this chapter, we've focused on dividing LLM computation into small steps: single-token generation. We saw this in Indirect Object Identification where we analyzed how GPT2-Small generates a single token. But modern reasoning models produce very long **chain-of-thought** traces, so we need to think about serialized computation over many tokens, not just over layers. This requires a new abstraction.

The [Thought Anchors](https://arxiv.org/abs/2506.19143) paper introduces this abstraction by splitting reasoning traces by **sentence**. Sentences are more coherent than tokens and correspond more closely to the actual reasoning steps. Some sentences matter a lot more than others for shaping the reasoning trajectory and the final answer; the authors call these **thought anchors**. They can be identified using black-box methods (resampling rollouts) or white-box methods (looking at / intervening on attention patterns). In these exercises, we'll work through both.

<img src="https://i.snipboard.io/PBoc9G.jpg" width="700">

We'll also explore the [Thought Branches paper](https://arxiv.org/abs/2510.27484) which extends these techniques to safety-relevant scenarios like blackmail, introducing metrics like **resilience** to distinguish genuine causal drivers from post-hoc rationalizations.

## Content & Learning Objectives

### 1️⃣ CoT Infrastructure & Sentence Taxonomy

> ##### Learning Objectives
>
> - Load and explore the MATH reasoning dataset structure (prompts, CoT traces, answers)
> - Implement sentence segmentation for chain-of-thought traces using regex patterns
> - Build both rule-based and LLM-based classifiers to categorize reasoning steps (problem_setup, active_computation, fact_retrieval, etc.)
> - Visualize how different sentence types are distributed across reasoning traces

### 2️⃣ Black-box Analysis

> ##### Learning Objectives
>
> - Understand **forced answer importance**: measuring impact by directly forcing model answers at specific steps
> - Implement **resampling importance**: measuring impact by having the model regenerate its own continuation from that point forward
> - Implement **counterfactual importance**: measuring impact by resampling then filtering to steps where the regenerated sentence is semantically different from the original
> - Learn when each metric is appropriate (causal vs correlational analysis)
> - Replicate key paper figures showing which reasoning steps are most critical for final answers

### 3️⃣ White-box Methods

> ##### Learning Objectives
>
> - Understand **receiver heads**: attention heads that aggregate information from reasoning steps into the final answer
> - Compute **vertical attention scores**: quantifying how much each token position attends to specific reasoning steps
> - Implement RoPE (Rotary Position Embeddings) to understand positional encoding in attention
> - Build attention suppression interventions to causally validate which attention patterns matter
> - Compare white-box attention metrics with black-box importance scores to validate mechanistic understanding

### 4️⃣ Thought Branches: Safety Applications

> ##### Learning Objectives
>
> - Apply thought anchor analysis to safety-critical scenarios (blackmail reasoning)
> - Measure how much reasoning steps influence safety-relevant outcomes (not just answer correctness)
> - Understand the **resilience metric**: how many times a sentence must be iteratively removed before it stays absent from the model's regeneration
> - Compare thought anchor patterns between mathematical reasoning (MATH benchmark) and safety reasoning (blackmail scenarios)

## Reading Material

Read at least the introduction and methods of the Thought Anchors paper before starting - the exercises follow its methodology closely. The Thought Branches paper is needed for Section 4.

- [Thought Anchors: Which LLM Reasoning Steps Matter?](https://arxiv.org/abs/2506.19143). Introduces the abstraction of treating sentences (not tokens) as units of reasoning in chain-of-thought traces, and identifies "thought anchors" - the sentences that most influence the final answer. Sections 1-3 of the exercises replicate the paper's black-box and white-box methods. Read the abstract, Section 1 (introduction) as well as 2 (explaining the methods for quantifying sentence importance) and 3 (explaining the sentence taxonomy). There's also a nice [interactive demo](https://thought-anchors.com/) showing thought anchors on example traces.
- [Thought Branches: Interpreting LLM Reasoning Requires Resampling](https://arxiv.org/abs/2510.27484). Extends thought anchor analysis to safety-relevant scenarios (e.g. blackmail reasoning), introducing the "resilience" metric. Mostly we just recommend skimming sections 1-3.
- [Principled Interpretability of Reward Hacking in Closed Frontier Models](https://www.lesswrong.com/posts/A67SbpTjuXEHK8Cvo/principled-interpretability-of-reward-hacking-in-closed) by Kroiz, Singh, Rajamanoharan & Nanda (MATS 9.0). Adapts the thought anchors resampling methodology to study reward hacking in closed frontier models (GPT-5, o3, Gemini 3 Pro), using agent actions as units of analysis rather than CoT sentences. Not needed for the exercises, but gives a sense of where the methodology is being applied.

## Setup code

Before running this code, you'll need to clone the [thought-anchors repo](https://github.com/interp-reasoning/thought-anchors). Make sure you're cloning it inside the `chapter4_alignment_science/exercises` directory.

```bash
cd chapter4_alignment_science/exercises
git clone https://github.com/interp-reasoning/thought-anchors.git
```

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter4_alignment_science"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install transformer_lens==2.17.0 einops jaxtyping openai

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else repo
    if Path(repo).exists()
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import json
import math
import os
import re
import sys
import time
import warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from types import MethodType
from typing import Any, Callable

import circuitsvis as cv
import einops
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
import transformers
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import disable_progress_bars, enable_progress_bars
from IPython.display import HTML, display
from jaxtyping import Float, Int
from openai import OpenAI
from plotly.subplots import make_subplots
from scipy import stats
from sentence_transformers import SentenceTransformer
from torch import Tensor
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria

warnings.filterwarnings("ignore")

thought_anchors_path = Path.cwd() / "thought-anchors"
assert thought_anchors_path.exists(), f"Please clone thought-anchors repo as {thought_anchors_path.resolve()!r}"

sys.path.append(str(thought_anchors_path))

t.set_grad_enabled(False)

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
dtype = t.bfloat16

# Make sure exercises are in the path
chapter = "chapter4_alignment_science"
section = "part3_interpreting_reasoning_models"
repo = "ARENA_3.0"
root_dir = next(p for p in Path.cwd().parents if p.name == repo)
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part3_interpreting_reasoning_models.tests as tests
import part3_interpreting_reasoning_models.utils as utils

MAIN = __name__ == "__main__"

You'll need to create an `.env` file in the `chapter4_alignment_science/exercises` directory before running the next cell.

In [ ]:
# Load .env file
env_path = exercises_dir / ".env"
assert env_path.exists(), "Please create a .env file with your API keys"

load_dotenv(dotenv_path=str(env_path))

# Setup OpenRouter client (works with both Claude and OpenAI models)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Please set OPENROUTER_API_KEY in your .env file"

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

Now let's define some constants:

In [ ]:
print(f"Using device: {device}")

# 1️⃣ CoT Infrastructure & Sentence Taxonomy

> ##### Learning Objectives
>
> - Load and explore the MATH reasoning dataset structure (prompts, CoT traces, answers)
> - Implement sentence segmentation for chain-of-thought traces using regex patterns
> - Build both rule-based and LLM-based classifiers to categorize reasoning steps (problem_setup, active_computation, fact_retrieval, etc.)
> - Visualize how different sentence types are distributed across reasoning traces

In this section, we'll build infrastructure to analyze chain-of-thought reasoning by breaking it into meaningful units. When models generate long reasoning traces, we need to decide what the right "units of reasoning" are.

Tokens are too granular: a single reasoning step like "Let's calculate the area: $\pi r^2$" spans multiple tokens, and splitting it apart loses semantic meaning. Sentences are a more natural unit. They correspond to complete thoughts, they're granular enough to identify specific important steps, and they align with how humans chunk reasoning.

By splitting CoT traces into sentences and categorizing them (calculations, lookups, restatements, etc.), we can ask: which types are most critical for reaching the correct answer? This sentence-level analysis sets us up for the black-box and white-box methods in later sections.

## Model Setup & Dataset Inspection

We'll start by setting a few constants (the purpose of these will become clear as we go through the exercises):

In [ ]:
# Configuration
MODEL_NAME_1B = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
MODEL_NAME_8B = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
MODEL_NAME_14B = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
DATASET_NAME = "uzaymacar/math-rollouts"
BLACKMAIL_DATASET_NAME = "uzaymacar/blackmail-rollouts"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
SIMILARITY_THRESHOLD = 0.8

Next, we'll load an embedding model. Embedding models take in text and output a vector - we'll use them to measure similarity between sentences, so we can find motifs in our reasoning traces.

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(embedding_model)

To understand how embedding models work, let's look at cosine similarities between a few example sentences:

In [ ]:
prompts = [
    "Wait, I think I made an error in my reasoning and need to backtrack",
    "Hold on, I believe I made a mistake in my logic and should reconsider",
    "After careful analysis, I've determined the correct answer is 42",
    "Time is an illusion. Lunchtime doubly so.",
]
labels = [x[:35] + "..." for x in prompts]

embedding = embedding_model.encode(prompts)
cosine_sims = embedding @ embedding.T

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cosine_sims, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
plt.colorbar(im, label="Cosine Similarity")
plt.title("Sentence Embedding Similarity")
plt.tight_layout()
plt.show()

We can also load the dataset that the paper's authors open-sourced. The dataset is very large, but the authors provide the structure on the [HuggingFace page](https://huggingface.co/datasets/uzaymacar/math-rollouts), so we can use `huggingface_hub` to load just the data we want. We'll inspect it more shortly.

In [ ]:
PROBLEM_ID = 4682

path = f"deepseek-r1-distill-llama-8b/temperature_0.6_top_p_0.95/correct_base_solution/problem_{PROBLEM_ID}/base_solution.json"
local_path = hf_hub_download(repo_id=DATASET_NAME, filename=path, repo_type="dataset")

with open(local_path, "r") as f:
    problem_data = json.load(f)

print("Keys in problem data:", list(problem_data.keys()))
print(f"\nProblem prompt:\n{problem_data['prompt']}")

## Sentence Splitting

First, we'll need to split our CoT traces into sentences based on punctuation and paragraph breaks. We'll also need to handle special tokens like `<think>`. We've provided the function `split_solution_into_chunks` below, which implements the following rules:

- The `<think> ... </think>` tags should be removed
- You should split on sentences (i.e. ending in any of `.`, `!`, `?`, or newlines), and characters like `:`
- You should split on periods `.` unless they are decimal numbers e.g. `x.y` or a numbered list e.g. `\n1.`
- No chunk should have length less than 10: if so, then merge it with the next chunk
- Each chunk should be stripped of whitespace

Read through the function and run the test cases below to verify it works on basic inputs.

### Recommended reading: edge cases in sentence splitting

After running the test cases, try to come up with your own input where this function breaks down or splits into chunks in a counter-intuitive way. Some things to consider: what happens with URLs, abbreviations (e.g. "Dr. Smith"), LaTeX expressions (e.g. `$3.14$`), or ellipses (`...`)? Once you've found an edge case, see if you can modify the function to handle it.

In [ ]:
def split_solution_into_chunks(text: str) -> list[str]:
    """Split solution into sentence-level chunks."""
    # Remove thinking tags
    if "<think>" in text:
        text = text.split("<think>")[1]
    if "</think>" in text:
        text = text.split("</think>")[0]
    text = text.strip()

    # Replace "." characters which we don't want to split on
    text = re.sub(r"(\d)\.(\d)", r"\1<DECIMAL>\2", text)  # e.g. "4.5" -> "4<DECIMAL>5"
    text = re.sub(r"\n(\d)\.(\s)", r"\n\1<DECIMAL>\2", text)  # e.g. "\n1. " -> "\n1<DECIMAL> "

    # Split on sentence endings, combining endings with previous chunk
    sentences = re.split(r"([!?:\n]|(?<!\n\d)\.)", text)
    chunks = []
    for i in range(0, len(sentences) - 1, 2):
        chunks.append((sentences[i] + sentences[i + 1]).replace("\n", " "))

    # Replace <DECIMAL> back with "."
    chunks = [re.sub(r"<DECIMAL>", ".", c) for c in chunks]

    # Merge chunks that are too short
    if not chunks:
        return []
    merged = [chunks[0]]
    for c in chunks[1:]:
        if len(merged[-1]) < 10:
            merged[-1] += c
        else:
            merged.append(c)
    return [c.strip() for c in merged if c.strip()]


test_cases = [
    (
        "<think>First, I understand the problem. Next, I'll solve for x. Finally, I verify!</think>",
        ["First, I understand the problem.", "Next, I'll solve for x.", "Finally, I verify!"],
    ),
    (
        "<think>Let me break this down: 1. Convert to decimal. 2. Calculate log. 3. Apply formula.</think>",
        ["Let me break this down:", "1. Convert to decimal.", "2. Calculate log.", "3. Apply formula."],
    ),
    (
        "<think>The answer is 42. Done.</think>",
        ["The answer is 42.", "Done."],
    ),
]

for input_text, expected_chunks in test_cases:
    chunks = split_solution_into_chunks(input_text)
    assert chunks == expected_chunks, f"Expected {expected_chunks}, got {chunks}"

print("All tests passed!")

## Sentence Categorization

The paper uses a taxonomy of 8 categories: Problem Setup (parsing/rephrasing the problem), Plan Generation (stating a plan of action), Fact Retrieval (recalling facts, formulas, problem details), Active Computation (algebra, calculations, manipulations), Uncertainty Management (expressing confusion, re-evaluating, backtracking), Result Consolidation (aggregating intermediate results), Self Checking (verifying previous steps), and Final Answer Emission (explicitly stating the final answer).

We define these below:

In [ ]:
CATEGORIES = {
    "problem_setup": "Problem Setup",
    "plan_generation": "Plan Generation",
    "fact_retrieval": "Fact Retrieval",
    "active_computation": "Active Computation",
    "uncertainty_management": "Uncertainty Management",
    "result_consolidation": "Result Consolidation",
    "self_checking": "Self Checking",
    "final_answer_emission": "Final Answer Emission",
    "unknown": "Unknown",
}

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "4.3.14"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part3_interpreting_reasoning_models.solutions import categorize_sentences_heuristic, generate_response, calculate_answer_importance, calculate_counterfactual_importance, precompute_rollout_embeddings, precompute_rollout_embeddings, get_resampled_rollouts, extract_attention_matrix, get_vertical_scores_simple, get_vertical_scores, find_receiver_heads, compute_receiver_head_scores, rotate_half


### Exercise - compute suppressed attention

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-25 minutes on this exercise
> ```

Now you'll implement the core logic for attention suppression. `compute_suppressed_attention` takes query and key states and computes attention weights, but with specific token ranges masked out (set to $-\infty$ before softmax, so they get zero attention weight after softmax).

How it works: (1) compute attention scores $\text{scores} = QK^T / \sqrt{d_k}$, (2) before softmax, mask specific token positions by setting their scores to $-\infty$ (minimum float value), (3) apply the attention mask if provided, (4) apply softmax. Setting scores to $-\infty$ before softmax ensures they become 0 after softmax, effectively removing those positions from the computation.

The key arguments are `token_ranges` (list of (start, end) tuples for which tokens to suppress), `heads_mask` (if provided, only suppress in specific heads), and `attention_mask` (standard causal/padding mask).

Hint: use `t.finfo(attn_weights.dtype).min` to get the minimum value for the tensor's dtype.

In [ ]:
def compute_suppressed_attention(
    query_states: Float[Tensor, "batch heads seq head_dim"],
    key_states: Float[Tensor, "batch heads seq head_dim"],
    token_ranges: list[tuple[int, int]],
    head_dim: int,  # for scaling attention weights by sqrt of!
    query_len: int,  # in case token_ranges exceed query length
    attention_mask: Float[Tensor, "batch 1 seq seq"] | None = None,
    heads_mask: Int[Tensor, " heads"] | None = None,  # shape (num_heads,) with True for heads to suppress
) -> Float[Tensor, "batch heads seq seq"]:
    # YOUR CODE HERE - compute attention scores, apply suppression, apply attention mask, then softmax


tests.test_compute_suppressed_attention(compute_suppressed_attention)

<details><summary>Solution</summary>

```python
def compute_suppressed_attention(
    query_states: Float[Tensor, "batch heads seq head_dim"],
    key_states: Float[Tensor, "batch heads seq head_dim"],
    token_ranges: list[tuple[int, int]],
    head_dim: int,  # for scaling attention weights by sqrt of!
    query_len: int,  # in case token_ranges exceed query length
    attention_mask: Float[Tensor, "batch 1 seq seq"] | None = None,
    heads_mask: Int[Tensor, " heads"] | None = None,  # shape (num_heads,) with True for heads to suppress
) -> Float[Tensor, "batch heads seq seq"]:
    # Compute attention scores
    attn_weights = t.matmul(query_states, key_states.transpose(2, 3)) / math.sqrt(head_dim)

    # Apply suppression mask BEFORE softmax (key step!)
    for start, end in token_ranges:
        effective_start = min(start, query_len)
        effective_end = min(end, query_len)
        if effective_start < effective_end:
            mask_value = t.finfo(attn_weights.dtype).min
            if heads_mask is None:
                # Suppress in all heads
                attn_weights[..., effective_start:effective_end] = mask_value
            else:
                # Suppress only in specific heads
                attn_weights[:, heads_mask, :, effective_start:effective_end] = mask_value

    # Apply attention mask if provided
    if attention_mask is not None:
        attn_weights = attn_weights + attention_mask

    # Softmax
    attn_weights = t.nn.functional.softmax(attn_weights, dim=-1, dtype=t.float32).to(query_states.dtype)
    return attn_weights


tests.test_compute_suppressed_attention(compute_suppressed_attention)
```
</details>

### Recommended reading: the full attention suppression hook

You don't need to implement this function from scratch. Instead, read through the implementation below and make sure you understand how each step works.

This code patches the forward method of Qwen's attention modules to use your `apply_rotary_pos_emb` and `compute_suppressed_attention` functions. It finds attention modules in each layer, stores original forward methods (for restoration later), creates masked forward functions that project to Q/K/V, apply RoPE, compute suppressed attention, and project output, then replaces the forward methods using `MethodType`.

A few implementation details: `layer_to_heads` allows suppressing only specific heads in specific layers (`None` = all heads in all layers), the function uses closures (`create_masked_forward`) to capture layer-specific parameters, and `repeat_kv` handles grouped-query attention where key/value heads are fewer than query heads.

How to use it:

```python
# Apply suppression to tokens 10-20 in all heads
suppression_info = apply_qwen_attention_suppression(model, token_ranges=[(10, 20)])

# Run model with suppression active
output = model(inputs)

# Restore original behavior
remove_qwen_attention_suppression(model, suppression_info)
```

As you read, pay attention to:

1. How the two functions you implemented (`apply_rotary_pos_emb` and `compute_suppressed_attention`) are used within the masked forward pass
2. How the patching mechanism works (replacing `forward` methods via `MethodType`)
3. What role `token_ranges` and `layer_to_heads` play in controlling the suppression

In [ ]:
def repeat_kv(
    hidden_states: Float[Tensor, "batch kv_heads seq head_dim"], n_rep: int
) -> Float[Tensor, "batch heads seq head_dim"]:
    """Expands key/value tensors for grouped-query attention."""
    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)


def apply_qwen_attention_suppression(
    model,
    token_ranges: list[tuple[int, int]] | tuple[int, int],
    layer_to_heads: dict[int, list[int]] | None = None,
) -> dict[str, Any]:
    """
    Suppresses attention to specific token positions by replacing forward methods.

    Args:
        model: The model to apply suppression to
        token_ranges: Token range(s) to suppress - single tuple or list of tuples
        layer_to_heads: Dict mapping layer indices to lists of head indices to suppress (None = all heads)

    Returns:
        Dict with 'original_forwards' for restoration

    Adapted from thought-anchors: whitebox-analyses/pytorch_models/hooks.py
    """
    # Normalize token_ranges to list of tuples
    if isinstance(token_ranges, tuple):
        token_ranges = [token_ranges]

    # Find rotary embedding module
    rotary_emb_module = None
    if hasattr(model, "model") and hasattr(model.model, "rotary_emb"):
        rotary_emb_module = model.model.rotary_emb

    # Find attention modules to patch
    target_modules = []
    for name, module in model.named_modules():
        if name.startswith("model.layers") and name.endswith("self_attn"):
            try:
                layer_idx = int(name.split(".")[2])
                if layer_to_heads is None or layer_idx in layer_to_heads:
                    if all(hasattr(module, attr) for attr in ["config", "q_proj", "k_proj", "v_proj", "o_proj"]):
                        target_modules.append((name, module, layer_idx))
            except (IndexError, ValueError):
                continue

    if not target_modules:
        print("Warning: No Qwen attention modules found to patch")
        return {"original_forwards": {}}

    # Store original forward methods
    original_forwards = {}

    # Create and apply masked forward functions
    for name, attn_module, layer_idx in target_modules:
        original_forwards[name] = attn_module.forward
        heads_mask = layer_to_heads[layer_idx] if layer_to_heads is not None else None

        # Create masked forward function
        def create_masked_forward(orig_forward, layer_idx, rotary_ref, heads_mask):
            def masked_forward(
                self,
                hidden_states: t.Tensor,
                attention_mask: t.Tensor | None = None,
                position_ids: t.LongTensor | None = None,
                past_key_value: tuple[t.Tensor] | None = None,
                output_attentions: bool = False,
                use_cache: bool = False,
                cache_position: t.LongTensor | None = None,
                **kwargs,
            ) -> tuple[t.Tensor, t.Tensor | None]:
                bsz, q_len, _ = hidden_states.size()
                config = self.config
                device = hidden_states.device

                # Project to Q, K, V
                query_states = self.q_proj(hidden_states)
                key_states = self.k_proj(hidden_states)
                value_states = self.v_proj(hidden_states)

                # Reshape for multi-head attention
                num_heads = config.num_attention_heads
                head_dim = config.hidden_size // num_heads
                num_key_value_heads = config.num_key_value_heads
                num_key_value_groups = num_heads // num_key_value_heads

                query_states = query_states.view(bsz, q_len, num_heads, head_dim).transpose(1, 2)
                key_states = key_states.view(bsz, q_len, num_key_value_heads, head_dim).transpose(1, 2)
                value_states = value_states.view(bsz, q_len, num_key_value_heads, head_dim).transpose(1, 2)

                # Apply RoPE
                if position_ids is None:
                    position_ids = t.arange(0, q_len, dtype=t.long, device=device).unsqueeze(0)

                if rotary_ref is not None and callable(rotary_ref):
                    cos, sin = rotary_ref(value_states, position_ids=position_ids)
                    query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin, position_ids)

                # Repeat K/V for grouped-query attention
                key_states = repeat_kv(key_states, num_key_value_groups)
                value_states = repeat_kv(value_states, num_key_value_groups)

                attn_weights = compute_suppressed_attention(
                    query_states,
                    key_states,
                    token_ranges,
                    head_dim,
                    q_len,
                    attention_mask,
                    heads_mask,
                )

                # Apply attention to values
                attn_output = t.matmul(attn_weights, value_states)

                # Reshape and project output
                attn_output = attn_output.transpose(1, 2).contiguous()
                attn_output = attn_output.reshape(bsz, q_len, config.hidden_size)
                attn_output = self.o_proj(attn_output)

                return attn_output, attn_weights if output_attentions else None

            return masked_forward

        attn_module.forward = MethodType(
            create_masked_forward(attn_module.forward, layer_idx, rotary_emb_module, heads_mask), attn_module
        )

    return {"original_forwards": original_forwards}


def remove_qwen_attention_suppression(model, suppression_info: dict[str, Any]):
    """Restores original forward methods after attention suppression."""
    original_forwards = suppression_info.get("original_forwards", {})
    if not original_forwards:
        return

    for name, module in model.named_modules():
        if name in original_forwards:
            module.forward = original_forwards[name]

#### Comprehension questions

Now that you've read through the implementation, try answering these questions to check your understanding.

**1. Why do we need to apply rotary position embeddings (RoPE) before computing attention scores?**

<details><summary>Answer</summary>

Rotary position embeddings encode relative position information into the query and key vectors. Without RoPE, the attention scores would have no awareness of token positions, so the model wouldn't know which tokens come before or after others. RoPE must be applied before computing attention because the attention pattern depends on the relative positions of the query and key tokens. If we skipped this step, the suppression mask would be applied to position-unaware attention scores, which wouldn't match the model's normal behavior.

</details>

**2. What would happen if we suppressed ALL sentence tokens rather than just the target sentence?**

<details><summary>Answer</summary>

Suppressing all sentence tokens would set all attention weights to negative infinity before softmax, leaving only the attention to non-sentence tokens (e.g. special tokens or whitespace). The model's output would become essentially meaningless since almost all semantic content is carried by sentence tokens. The point of suppressing a single target sentence is to isolate its causal contribution: we want to measure how much removing attention to *that specific sentence* changes the output, while keeping the rest of the context intact.

</details>

**3. Why do we compute KL divergence rather than just checking if the top predicted token changes?**

<details><summary>Answer</summary>

Checking only the top predicted token is a very coarse measure. A sentence could substantially shift the probability distribution (e.g. redistributing probability mass among the top-5 tokens) without changing which token has the highest probability. KL divergence captures the full distributional shift, including subtle changes in confidence and probability rankings. This is especially important for thought anchors: a sentence might not change the single most likely next token, but it could significantly affect the model's uncertainty and the tail of the distribution, which matters for downstream reasoning over many token positions.

</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# Wrap tests.test_compute_suppressed_attention so a passing test fires the beacon.
try:
    _dd_orig = tests.test_compute_suppressed_attention
    def _dd_wrapped(*args, **kwargs):
        result = _dd_orig(*args, **kwargs)
        _dd_report_complete()
        return result
    tests.test_compute_suppressed_attention = _dd_wrapped
except AttributeError:
    print('[Delta Drills] no matching test function — call _dd_report_complete() manually when done.')
